<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 points)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [1]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa,
                          beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
  age <- runif(sample_size, age_lo, age_hi)
  epsilon_x <- rnorm(sample_size, mean = 0, sd = sd_x)
  epsilon_c <- rnorm(sample_size, mean = 0, sd = sd_c)
  
  x <- beta_xa * age + beta_x0 + epsilon_x
  c <- beta_ca * age + beta_cx * x + beta_c0 + epsilon_c

      return(data.frame(age=age,x=x,c=c)) # it's actually bad form to have a variable named "c" in R, my bad...
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,125.9807,110.43061,472.9583
2,172.1844,101.15432,502.8793
3,131.4637,117.10039,499.4796
4,113.9560,26.60062,190.4848
5,152.5084,146.09782,482.2208
6,162.1389,30.25063,257.0601


---
## 2. `run_analysis()` function (2 pts)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [3]:
library(mediation)

run_analysis <- function(data) {
  fitM <- lm(x ~ age, data = data)
  fitY <- lm(c ~ x + age, data = data)
  fitMed <- mediate(fitM, fitY, treat = "age", mediator = "x")
  return(c(fitMed$d0, fitMed$d0.p, fitMed$z0, fitMed$z0.p))
}

run_analysis(dat)

[1] 1.7283412 0.0040000 0.7411972 0.0120000

---
## 3. `repeat_analysis()` function (3 pts)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [4]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    # Start simulating
    for (i in 1:num_simulations) {
      data <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
      simouts[i,] <- run_analysis(data)

    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] <= alpha)
    ADE_cov =  mean(simouts[,4] <= alpha)

    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [5]:
repeat_analysis(num_simulations = 10, alpha = 0.01, sample_size = 100,
                age_lo = 80, age_hi = 200, beta_xa = 0.5, beta_x0 = -5,
                sd_x = 50, beta_ca = 0.8, beta_cx = 3, beta_c0 = 10, sd_c = 85)

$ACME_cov
[1] 0.9

$ADE_cov
[1] 0.8

---
## 4. Testing different sample sizes (2 pts)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [6]:
sample_sizes <- c(50, 75, 100, 125, 150)
results <- list()

for (i in 1:length(sample_sizes)) {
  results[[i]] <- repeat_analysis(num_simulations = 10, alpha = 0.01,
                                  sample_size = sample_sizes[i],
                                  age_lo = 80, age_hi = 200, beta_xa = 0.5,
                                  beta_x0 = -5, sd_x = 50, beta_ca = 0.8,
                                  beta_cx = 3, beta_c0 = 10, sd_c = 85)
}

Print your results.

In [7]:
for (i in 1:length(sample_sizes)) {
  cat("Sample size:", sample_sizes[i], 
      "| ACME power:", results[[i]]$ACME_cov, 
      "| ADE power:", results[[i]]$ADE_cov, "\n")
}

Sample size: 50 | ACME power: 0.7 | ADE power: 0.5 
Sample size: 75 | ACME power: 0.4 | ADE power: 0.8 
Sample size: 100 | ACME power: 0.8 | ADE power: 0.6 
Sample size: 125 | ACME power: 0.9 | ADE power: 0.9 
Sample size: 150 | ACME power: 1 | ADE power: 0.9 


## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> From the above results, we can see that a sample size of around 125-150 children would be needed to reliably detect both the ACME and ADE at an alpha level of 0.01. At smaller sample sizes like 50 or 75, the power is pretty inconsistent and low, meaning that there is a higher risk of failing to detect real effects. By the sample size reaching 125, both ACME and ADE power reached approximately to 0.9, which could be considered acceptable. And since there are only 10 simulations per sample size, these power estimates would also be pretty noisy, so more simulations can give us more stable estimates.

**Given** how we generated the data, why was the direct effect harder to detect than the mediated effect?
> ADE was harder because when we generate the data, much of age's influence on comprehension is channeled thorugh word reading experience (the mediated pathway). Since age affects experience (beta_xa = 0.5) and experience has an effect on comprehension (beta_cx = 3), this could make the indirect/mediated path quite large. So the direct effect of age on comprehension (beta_ca = 0.8) is comparatively smaller. Since the mediation model partitions the total effect into direct and indirect components, the direct effect thus has a weaker signal relative to the noise, making it harder to detect.

**DUE:** 5pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*